# Clase 8. Gradientes de política y métodos Actor-Crítico

**DIH DATalife — Curso de Reinforcement Learning (RL) desde cero**

Bienvenida o bienvenido a la clase ocho. Hasta ahora, tu agente aprendía cuánto vale cada acción y, después, actuaba eligiendo siempre la de mayor valor. Hoy cambias de enfoque: aprendes la política directamente, sin ese paso intermedio. Vas a entender por qué esto importa, cuándo actuar al azar es la mejor estrategia posible, y vas a programar REINFORCE, el algoritmo de gradiente de política más simple, sobre CartPole.

## Guion de la clase

1. De la función de valor a la política directa
2. Por qué a veces conviene actuar al azar
3. El truco de la función score
4. REINFORCE, paso a paso
5. Reducir la varianza: línea base y ventaja
6. Arquitectura Actor-Crítico
7. Código en vivo: REINFORCE en CartPole
8. Puntos clave y qué viene en la clase 9


## 1. De la función de valor a la política directa

*Guion de clase: recuerda en voz alta cómo funcionaba Deep Q-Network (DQN) en la clase siete, y pregunta a la clase qué pasaría si el número de acciones posibles fuera infinito.*

Deep Q-Network (DQN), que viste en la clase anterior, aprende cuánto vale cada acción en cada estado mediante una red neuronal, y después actúa eligiendo siempre la acción de mayor valor. Funciona muy bien, pero da un rodeo: primero aprende una función de valor completa y solo después, de forma indirecta, deriva una política a partir de ella.

Hoy ves una idea distinta: en vez de aprender cuánto vale cada acción, aprende directamente la política, la regla que decide qué hacer en cada estado. Piensa en un robot cuya mejor estrategia es, simplemente, "avanzar siempre": su política óptima es trivial, pero la función de valor que describe cuánto vale cada combinación de posición y obstáculos puede ser enormemente compleja. Si tu objetivo real es actuar bien, ¿por qué aprender primero ese rodeo complicado en vez de ir directo a la política? Este principio —evita resolver un problema más general del que necesitas para resolver el problema concreto que te interesa— se atribuye al estadístico Vladimir Vapnik, y es la motivación central de la clase de hoy.

Aprender la política directamente tiene, además, dos ventajas prácticas que DQN no ofrece con la misma naturalidad:

- **Espacios de acción continuos o muy grandes**: pensemos en el ángulo de un motor o la fuerza de un brazo robótico. DQN necesita calcular el máximo sobre todas las acciones posibles, algo inviable cuando hay infinitas. Un método basado en política, en cambio, simplemente emite la acción (o una distribución sobre acciones) sin necesitar ese máximo
- **Políticas genuinamente aleatorias**, no solo deterministas, algo que entiendes en el siguiente apartado


## 2. Por qué a veces conviene actuar al azar

*Guion de clase: plantea el laberinto como pregunta abierta antes de dar la respuesta. "¿Qué le pasa a un agente determinista aquí?"*

Imagina un laberinto con dos pasillos que, vistos desde dentro, parecen exactamente iguales: mismas paredes a izquierda y derecha, mismo aspecto. Un agente que solo sabe actuar de forma determinista puede quedar atrapado, yendo de un pasillo a otro en bucle infinito, porque desde su punto de vista ambos estados son indistinguibles. Una política estocástica, que en esos pasillos elige izquierda o derecha al azar, tarde o temprano escapa del bucle por pura probabilidad y llega a una zona donde sí puede distinguir dónde está.

Lo mismo pasa en juegos con información oculta. En piedra, papel o tijera, cualquier estrategia determinista es explotable de inmediato: tu rival solo necesita detectar el patrón. La única estrategia óptima es elegir cada opción con la misma probabilidad. En el póquer ocurre algo parecido: si tu forma de actuar con una mano ganadora es siempre la misma, un rival atento la aprende y la explota; el farol solo funciona si es probabilístico.

Una política, entonces, ya no es una regla fija "estado → acción" (una función determinista `a = μ(s)`), sino una distribución de probabilidad sobre las acciones dado el estado:

`π(a|s)`

Aprender esta distribución, en vez de una regla fija, es lo que hace posible representar comportamientos como los del laberinto o el póquer.


## 3. El truco de la función score

*Guion de clase: no te enredes en el desarrollo matemático completo. Transmite solo la idea intuitiva y la fórmula final, remarcando lo práctico de la consecuencia final (no hace falta modelar el entorno).*

Aquí surge un problema técnico. Quieres ajustar los parámetros de tu política, θ, para maximizar la recompensa esperada. Pero esa recompensa esperada depende de un proceso de muestreo aleatorio: la acción que tomas se saca de una distribución de probabilidad, y no puedes "derivar" directamente a través de un sorteo.

La solución se conoce como truco de la función score (o truco REINFORCE, o truco log-derivada). La idea intuitiva, sin el desarrollo matemático completo: en vez de intentar derivar el resultado de un muestreo, conviertes el problema en algo que sí sabes derivar, el logaritmo de la probabilidad de la acción que elegiste. El resultado es sorprendentemente simple de aplicar: para ajustar la política, calculas el gradiente del logaritmo de la probabilidad de cada acción que tomaste, y lo multiplicas por lo buena que resultó esa acción (su recompensa o retorno).

Dicho de forma llana: sube la probabilidad de las acciones que llevaron a una recompensa alta, y baja la probabilidad de las que llevaron a una recompensa baja o negativa. Cuanto mejor haya sido el resultado, más fuerte es el empujón hacia arriba.

**Un detalle práctico importante**: cuando descompones la probabilidad de una trayectoria completa, aparecen dos tipos de términos, los que dependen de la dinámica del entorno (qué estado sigue a cuál) y los que dependen solo de tu política. Al calcular el gradiente respecto a los parámetros de la política, los términos de la dinámica del entorno desaparecen, porque no dependen de θ. Esto significa que los métodos de gradiente de política no necesitan conocer ni modelar cómo funciona el entorno: les basta con poder actuar en él y observar qué pasa.


## 4. REINFORCE: el algoritmo más simple

*Guion de clase: escribe los cuatro pasos en la pizarra antes de mostrar código. Insiste en que "retorno" aquí es la suma de recompensas futuras del episodio completo, no un valor estimado por una red.*

Con esa idea ya tienes el algoritmo REINFORCE, la versión más básica de gradiente de política:

1. Inicializa los parámetros θ de la política, normalmente una red neuronal
2. Repite hasta que el rendimiento te convenza:
   - Ejecuta un episodio completo siguiendo la política actual y guarda estados, acciones y recompensas
   - Para cada paso del episodio, calcula el retorno (la suma de recompensas futuras, normalmente con descuento)
   - Calcula el gradiente: para cada acción tomada, multiplica el gradiente del logaritmo de su probabilidad por el retorno correspondiente
   - Actualiza los parámetros en la dirección de ese gradiente: `θ ← θ + α · gradiente`

Es un algoritmo elegante y fácil de programar. Pero tiene un problema serio: alta varianza. Cada episodio depende de decisiones aleatorias (qué acciones sacó el muestreo) y de la aleatoriedad propia del entorno, así que el retorno de un episodio puede ser muy distinto del retorno de otro episodio casi idéntico. Eso hace que las actualizaciones salten de un lado a otro y el entrenamiento sea lento e inestable. Lo vas a ver con tus propios ojos dentro de un rato, en la gráfica del código en vivo.


## 5. Reducir la varianza: línea base y ventaja

*Guion de clase: pregunta antes de explicar por qué restar un número constante no cambia la dirección del gradiente. Deja que alguien de la clase intente razonarlo.*

Aquí entra una idea muy elegante. Puedes restar del retorno una línea base, `b(s)`, un valor que depende solo del estado (no de la acción elegida), sin introducir ningún sesgo en el aprendizaje. La razón es que, al sumar sobre todas las acciones posibles, el efecto de una línea base constante se cancela matemáticamente: no cambia la dirección del gradiente, solo reduce su varianza.

La línea base más habitual es la propia función de valor de estado, `V(s)`: cuánta recompensa esperas obtener desde ese estado, siguiendo tu política actual. Al restarla del valor de acción `Q(s,a)`, obtienes la función de ventaja:

`A(s,a) = Q(s,a) − V(s)`

La ventaja responde a una pregunta muy concreta: ¿cuánto mejor (o peor) es esta acción concreta comparada con lo que harías en promedio en este estado? Una ventaja positiva empuja la probabilidad de esa acción hacia arriba; una ventaja negativa la empuja hacia abajo. Es una señal mucho más limpia que el retorno bruto, porque ya no arrastra la parte de la recompensa que depende solo del estado en el que estabas, sino solo la parte que depende de la decisión que tomaste.

En la práctica, no siempre necesitas dos redes separadas para Q y V. Puedes estimar la ventaja con el error de diferencia temporal (TD, por las siglas de "temporal difference"): `δ = r + γV(s') − V(s)`. Este error compara lo que esperabas obtener con lo que realmente obtuviste tras dar un paso, y es, en la práctica, una muestra de la ventaja.


## 6. Arquitectura Actor-Crítico

*Guion de clase: cuenta la analogía del chef y la persona experta en gastronomía antes de mostrar el pseudocódigo. Relaciona cada línea del pseudocódigo con su papel en la analogía.*

Esa última idea lleva directamente a los métodos Actor-Crítico. Imagina un chef (el actor) que decide qué plato cocinar, y una persona experta en gastronomía (el crítico) que prueba cada plato y le dice cuánto mejor o peor fue que lo esperado. El chef usa esa valoración para ajustar sus próximas decisiones.

En términos técnicos, el actor es la política `π_θ`, que decide qué acción tomar, y el crítico es una función de valor `V_ω`, que evalúa qué tan buena está resultando la actuación del actor. El crítico se entrena con técnicas de diferencia temporal, igual que la función de valor de DQN. El actor se entrena con el gradiente de política, usando el error TD del crítico como señal de ventaja. El esquema básico es este:

```
Inicializa la red del actor (política) con parámetros θ
Inicializa la red del crítico (valor) con parámetros ω

para cada episodio:
    inicializa el estado s
    para cada paso del episodio:
        muestrea una acción a ~ π_θ(a|s)
        ejecuta a, observa la recompensa r y el siguiente estado s'
        δ = r + γ · V_ω(s') − V_ω(s)          # error TD, usado como ventaja
        ω ← ω + α_crítico · δ · ∇_ω V_ω(s)     # actualiza el crítico
        θ ← θ + α_actor · δ · ∇_θ log π_θ(a|s) # actualiza el actor
        s ← s'
```

Este patrón, en el que el crítico aprende con cada paso (no hace falta esperar a que termine el episodio) y el actor usa esa señal para mejorar, se conoce como Advantage Actor-Critic (A2C), y su variante que ejecuta varios agentes en paralelo, cada uno con su propia copia del entorno, se llama Asynchronous Advantage Actor-Critic (A3C). Vuelves a encontrarte con A3C en el capítulo 12 del manual.

Existen evoluciones posteriores de estas ideas que vas a encontrar mencionadas más adelante en el curso, sin entrar en su detalle técnico hoy: Trust Region Policy Optimization (TRPO) y Proximal Policy Optimization (PPO), que limitan cuánto puede cambiar la política en cada actualización para evitar pasos demasiado bruscos, y Deep Deterministic Policy Gradient (DDPG), pensado para espacios de acción continuos.


## 7. Código en vivo: REINFORCE en CartPole

*Guion de clase: construye el código celda a celda, sin copiar y pegar el bloque entero. Antes de ejecutar la celda del bucle de entrenamiento, pregunta a la clase qué esperan ver en la gráfica final.*

Vas a implementar REINFORCE desde cero sobre CartPole, el entorno clásico en el que un carro debe mantener un palo en equilibrio moviéndose a izquierda o derecha. Todo el código funciona en Google Colab: usa PyTorch (ya preinstalado) y gymnasium (se instala en la primera celda). No necesitas GPU.

**Ejecuta cada celda en orden, de arriba abajo.**


### Paso 1 — Instala gymnasium

PyTorch ya viene preinstalado en Colab, así que solo te falta gymnasium, la librería de entornos.

In [ ]:
# Celda 1: instala gymnasium (PyTorch ya viene preinstalado en Colab)
!pip install gymnasium

### Paso 2 — Importa librerías y fija la semilla

*Guion de clase: recuerda por qué fijamos semillas — reproducibilidad, no azar real.*

Fijas la semilla de PyTorch y de NumPy para que el experimento sea reproducible: con la misma semilla, obtienes siempre la misma secuencia de números aleatorios.

In [ ]:
import gymnasium as gym
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt

# Fijamos semillas para que el experimento sea reproducible
SEED = 0
torch.manual_seed(SEED)
np.random.seed(SEED)

### Paso 3 — Define la red de la política

`RedPolitica` representa `π_θ(a|s)`: recibe el estado de CartPole (cuatro números —posición y velocidad del carro, ángulo y velocidad angular del palo—) y devuelve una distribución de probabilidad sobre las dos acciones posibles (empujar el carro a la izquierda o a la derecha). La capa `softmax` final convierte los `logits` de la red en probabilidades que suman uno.

In [ ]:
class RedPolitica(nn.Module):
    """Red neuronal pequeña que representa la política π_θ(a|s).
    Recibe el estado de CartPole (cuatro números: posición y velocidad del
    carro, ángulo y velocidad angular del palo) y devuelve una distribución
    de probabilidad sobre las dos acciones posibles (empujar el carro a la
    izquierda o a la derecha)."""

    def __init__(self, n_entradas, n_acciones, n_ocultas=128):
        super().__init__()
        self.red = nn.Sequential(
            nn.Linear(n_entradas, n_ocultas),
            nn.ReLU(),
            nn.Linear(n_ocultas, n_acciones),
        )

    def forward(self, estado):
        logits = self.red(estado)
        return torch.softmax(logits, dim=-1)

### Paso 4 — Calcula los retornos descontados

`calcular_retornos` implementa el "reward-to-go": en cada paso `t`, suma las recompensas futuras del episodio, cada una descontada por `gamma` elevado a la distancia en pasos. Recorrer la lista de recompensas de atrás hacia adelante te permite reutilizar el cálculo del paso siguiente en cada paso anterior, en vez de recalcular la suma completa cada vez.

In [ ]:
def calcular_retornos(recompensas, gamma=0.99):
    """Calcula el retorno 'reward-to-go' en cada paso del episodio:
    G_t = r_t + gamma*r_{t+1} + gamma^2*r_{t+2} + ...
    Se recorre la lista de atrás hacia adelante para reutilizar el cálculo
    del paso siguiente en cada paso anterior."""
    retornos = np.zeros(len(recompensas))
    G = 0.0
    for t in reversed(range(len(recompensas))):
        G = recompensas[t] + gamma * G
        retornos[t] = G
    return retornos

### Paso 5 — Recolecta un episodio completo

`ejecutar_episodio` sigue la política actual desde el reinicio del entorno hasta que termina el episodio. En cada paso, muestrea una acción de la distribución `π_θ(a|s)` (con `torch.distributions.Categorical`) y guarda su log-probabilidad, que necesitas después para el gradiente de política.

In [ ]:
def ejecutar_episodio(env, politica):
    """Recolecta un episodio completo siguiendo la política actual.
    Devuelve la log-probabilidad de cada acción tomada (necesaria para el
    gradiente de política) y la recompensa obtenida en cada paso."""
    log_probs = []
    recompensas = []
    estado, _ = env.reset()
    terminado = False
    truncado = False

    while not (terminado or truncado):
        estado_tensor = torch.FloatTensor(estado).unsqueeze(0)
        probs = politica(estado_tensor)
        distribucion = torch.distributions.Categorical(probs)
        accion = distribucion.sample()

        log_probs.append(distribucion.log_prob(accion))
        estado, recompensa, terminado, truncado, _ = env.step(accion.item())
        recompensas.append(recompensa)

    return log_probs, recompensas

### Paso 6 — Configura el entorno y los hiperparámetros

Creas el entorno CartPole-v1, instancias la red de política y el optimizador Adam, y fijas los dos hiperparámetros centrales: el número de episodios de entrenamiento y `GAMMA`, el factor de descuento.

In [ ]:
env = gym.make("CartPole-v1")
n_entradas = env.observation_space.shape[0]
n_acciones = env.action_space.n

politica = RedPolitica(n_entradas, n_acciones)
optimizador = optim.Adam(politica.parameters(), lr=1e-2)

N_EPISODIOS = 500
GAMMA = 0.99

historial_recompensas = []

### Paso 7 — El bucle de entrenamiento

*Guion de clase: detente en la línea de la pérdida. Pregunta: "¿por qué hay un signo negativo?" antes de leer el comentario en voz alta.*

Este es el corazón de REINFORCE. Para cada episodio:

1. Recolectas la experiencia con `ejecutar_episodio`
2. Calculas el retorno de cada paso con `calcular_retornos`
3. Construyes la pérdida: `-sum(log_prob(a_t|s_t) * G_t)`. El signo negativo convierte el ascenso de gradiente que quieres (maximizar la recompensa esperada) en el descenso de gradiente que optimiza PyTorch por defecto
4. Retropropagas y actualizas los parámetros de la política con el optimizador

**Ejecuta esta celda y observa cómo va subiendo la recompensa media cada 50 episodios.** En CartPole-v1, un episodio dura como máximo 500 pasos, así que una recompensa media cercana a ese número indica que el palo se mantiene en equilibrio casi todo el tiempo permitido.

In [ ]:
for episodio in range(N_EPISODIOS):
    log_probs, recompensas = ejecutar_episodio(env, politica)
    retornos = calcular_retornos(recompensas, GAMMA)
    retornos_tensor = torch.FloatTensor(retornos)

    # Pérdida del gradiente de política: -sum(log_prob(a_t|s_t) * G_t)
    # El signo negativo convierte el ASCENSO de gradiente que queremos
    # (maximizar la recompensa esperada) en el DESCENSO de gradiente que
    # optimiza PyTorch por defecto.
    perdida = torch.stack(
        [-log_prob * G for log_prob, G in zip(log_probs, retornos_tensor)]
    ).sum()

    optimizador.zero_grad()
    perdida.backward()
    optimizador.step()

    historial_recompensas.append(sum(recompensas))

    if (episodio + 1) % 50 == 0:
        media_reciente = np.mean(historial_recompensas[-50:])
        print(f"Episodio {episodio + 1}: recompensa media (últimos 50) = {media_reciente:.1f}")

env.close()

### Paso 8 — Grafica la recompensa por episodio

La curva fina muestra la recompensa episodio a episodio; la curva gruesa es una media móvil que suaviza el ruido para que veas mejor la tendencia.

**Ejecuta esta celda y fíjate en los altibajos entre episodios cercanos.**

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(historial_recompensas, alpha=0.4, label="Recompensa por episodio")

ventana = 20  # media móvil para ver la tendencia sin el ruido episodio a episodio
media_movil = np.convolve(historial_recompensas, np.ones(ventana) / ventana, mode="valid")
plt.plot(range(ventana - 1, len(historial_recompensas)), media_movil, label=f"Media móvil ({ventana} episodios)")

plt.xlabel("Episodio")
plt.ylabel("Recompensa total")
plt.title("REINFORCE en CartPole-v1")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

### Lectura del resultado

*Guion de clase: para aquí y pregunta a la clase qué curva esperaban frente a lo que ven. Conecta explícitamente esto con el problema de varianza del apartado cuatro.*

Al ejecutarlo ves algo típico de REINFORCE: la curva de recompensa sube, pero con muchos altibajos entre episodios cercanos, justo el problema de varianza que ya conoces del apartado cuatro. Ese vaivén es la razón de ser de la línea base y la ventaja del apartado cinco: en el notebook de práctica de esta clase vas a comprobar tú misma o tú mismo cuánto ayuda a suavizar esta curva.


## Puntos clave

- Los métodos de gradiente de política aprenden directamente la política, sin pasar por una función de valor completa, lo que encaja mejor con espacios de acción continuos o muy grandes
- Una política estocástica, `π(a|s)`, es necesaria en estados ambiguos y en juegos con información oculta, donde una regla determinista sería explotable
- El truco de la función score convierte un problema de derivar a través de un muestreo aleatorio en algo calculable: el gradiente del logaritmo de la probabilidad de la acción, multiplicado por lo buena que resultó
- REINFORCE aplica esa idea de forma directa, pero sufre de varianza alta porque depende de retornos completos de episodios ruidosos
- Restar una línea base (habitualmente `V(s)`) reduce la varianza sin introducir sesgo, y da lugar a la función de ventaja `A(s,a) = Q(s,a) − V(s)`
- La arquitectura Actor-Crítico combina un actor (la política) con un crítico (una función de valor) que evalúa cada paso y guía la actualización del actor con el error TD
- A2C, A3C, PPO, TRPO y DDPG son evoluciones de estas ideas que vas a ir encontrando en próximas clases, sin necesidad de dominar su detalle técnico todavía


## Antes de la próxima clase

**Abre el notebook de práctica de esta clase y completa los dos ejercicios sobre REINFORCE.** Vas a programar tú misma o tú mismo el cálculo de los retornos descontados y la pérdida de REINFORCE desde cero, y vas a añadir una línea base para comprobar en primera persona cuánto reduce la varianza del entrenamiento.

En la clase 9 cambias de familia de métodos: vas a ver qué pasa cuando el agente construye o usa un modelo del entorno para planificar por adelantado, con técnicas como Dyna-Q y la búsqueda en árbol de Monte Carlo (Monte Carlo Tree Search).
